# 세 프레임으로 YOLO26n-pose 시험 학습하기

수행자 박스, 무릎, 발목을 YOLO 라벨로 바꾼 뒤 원본에 다시 그려 확인한다. a08 29번 한 장으로 학습하고 b02 36번·c04 78번 두 장으로 검증한다. 처음에는 **모든 셀 실행**으로 그림까지 확인할 수 있다. 학습을 시작하는 스위치는 4단계에 있다.

### 1. 사람별 분할과 원본 라벨 확인

선수 본인을 기준으로 a08·b02는 오른쪽, c04는 왼쪽이다. 무릎 관절선과 발목 복사뼈를 둘 다 직접 짚고 전신 박스까지 저장한 세 장을 사용한다.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from wlcv.config import DEVICE, ROOT
from wlcv.data.pose_dataset import export_pose_dataset, load_reviewed_frames
from wlcv.evaluation.pose_label_review import review_pose_labels
from wlcv.training.pose_smoke import train_pose_smoke

frames = load_reviewed_frames()
display(pd.DataFrame([
    {"영상": f.video.stem, "프레임": f.frame, "분할": f.split, "가까운 쪽": f.side,
     "원본 박스": f.box, "원본 무릎": f.points["knee"], "원본 발목": f.points["ankle"]}
    for f in frames
]))

### 2. 원본 이미지와 YOLO 라벨 저장

수행자 한 명이 TXT 한 줄이다. 앞의 다섯 값은 사람 클래스, 박스 중심 x·y, 너비·높이다. 이어서 COCO 순서대로 17개 관절의 x·y·가시성을 적는다. x와 너비는 원본 너비로, y와 높이는 원본 높이로 나눠 0~1 좌표를 만든다. [Ultralytics Pose 데이터 형식](https://docs.ultralytics.com/datasets/pose/)

직접 확인한 무릎·발목은 가시성 2다. 추정·미판독·반대쪽 관절은 모두 `0 0 0`이다. 이 점들은 위치 오차 계산에서 빠지지만 학습 중 관절 가시성의 0 표적으로는 쓰인다. 이 시험에서 전체 관절의 정확도를 해석하기 어려운 이유다.

In [ ]:
data_yaml = export_pose_dataset(frames, ROOT / "outputs/pose_smoke")
print("데이터 설정 파일", data_yaml)
print(data_yaml.read_text())
label = data_yaml.parent / "labels/train" / f"{frames[0].stem}.txt"
print("a08의 YOLO 라벨", label.read_text().strip())

### 3. 저장된 TXT를 원본에 다시 그리기

윗줄은 원본 전체, 아랫줄은 같은 프레임의 수행자 확대다. 초록 박스가 수행자를 감싸는지, 청록 십자가 무릎에, 노란 마름모가 발목에 있는지 확인한다. 표는 TXT를 픽셀로 되돌렸을 때 CSV와 가장 많이 차이 난 좌표의 오차다. 어느 한 좌표라도 0.01픽셀을 넘거나 좌우가 바뀌면 검사를 멈춘다.

In [ ]:
verified_yaml = None
figure, errors = review_pose_labels(data_yaml, frames)
preview = data_yaml.parent / "label_review.png"
figure.savefig(preview, dpi=140)
plt.show()
display(errors.style.format({column: "{:.6f}" for column in errors.filter(like="오차")}))
print("확인 그림", preview)
verified_yaml = data_yaml

### 4. 확인한 데이터로 1 epoch 실행

그림과 표를 확인한 뒤 아래 셀의 `RUN_TRAINING`을 `True`로 바꾸고 **이 셀만** 실행한다. `batch=1`은 학습 한 장을 한 배치로 읽는 설정이고, `workers=0`은 Mac 노트북의 현재 프로세스에서 데이터를 읽는 설정이다. 학습 장치는 프로젝트 설정을 따르며 Apple Silicon에서 MPS를 사용할 수 있으면 기본으로 선택한다.

완료되면 이번 실행의 `best.pt`와 지표 한 행이 표시된다. 여기서 확인할 것은 학습, 검증, 가중치 저장까지 한 번 이어지는지다. 두 검증 프레임의 mAP로 성능 개선을 판단하기에는 표본이 작다.

In [ ]:
RUN_TRAINING = False

if RUN_TRAINING:
    if verified_yaml != data_yaml:
        raise ValueError("3단계를 실행해 이번에 변환한 라벨을 먼저 확인하세요.")
    run_dir = train_pose_smoke(verified_yaml, device=DEVICE)
    print("실행 폴더", run_dir)
    print("최고 가중치", run_dir / "weights/best.pt")
    print("마지막 가중치", run_dir / "weights/last.pt")
    display(pd.read_csv(run_dir / "results.csv").round(4))
else:
    print(f"선택한 장치 {DEVICE} · 그림 확인 후 이 셀에서 학습을 시작하세요.")